#### REAL-TIME GESTURE DETECTOR

### 0. Data Collection

In [1]:
import cv2
import mediapipe as mp
import numpy as np
import os
import time
from datetime import datetime

GESTURES = [
    "jump", "crouch", "move_left", "move_right",
    "high_punch", "low_punch", "strong_kick", "high_kick",
    "hit_combo", "neutral"
]

TARGET_FPS = 60
FRAME_INTERVAL = 1.0 / TARGET_FPS
FRAMES_PER_CLIP = 60   # 1 second at 60 FPS
CLIPS_PER_GESTURE = 3

OUTPUT_DIR = "karate_data_trial"
os.makedirs(OUTPUT_DIR, exist_ok=True)

for g in GESTURES:
    os.makedirs(os.path.join(OUTPUT_DIR, g), exist_ok=True)

mp_pose = mp.solutions.pose
pose = mp_pose.Pose()

cap = cv2.VideoCapture(0)

def capture_clip(label, clip_no):
    keypoints_list = []
    print(f"\nPrepare for {label} clip {clip_no+1}")

    for i in range(3, 0, -1):
        print(f"Starting in {i}...")
        cv2.waitKey(1000)

    print("Recording...")
    frame_count = 0

    while frame_count < FRAMES_PER_CLIP:
        start_time = time.time()

        ret, frame = cap.read()
        if not ret:
            break

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        result = pose.process(rgb)

        if result.pose_landmarks:
            kps = [[lm.x, lm.y, lm.z] for lm in result.pose_landmarks.landmark]
            keypoints_list.append(kps)
        else:
            keypoints_list.append(np.zeros((33, 3)))

        cv2.putText(frame, f"{label} clip {clip_no+1}", (10, 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        cv2.imshow("Recording", frame)

        frame_count += 1

        # ---- FPS CONTROL ----
        elapsed = time.time() - start_time
        sleep_time = FRAME_INTERVAL - elapsed
        if sleep_time > 0:
            time.sleep(sleep_time)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    np.save(os.path.join(OUTPUT_DIR, label, f"{timestamp}.npy"),
            np.array(keypoints_list))

    print(f"Saved {label} clip {clip_no+1}")

# Collect all gestures
for g in GESTURES:
    for clip_no in range(CLIPS_PER_GESTURE):
        capture_clip(g, clip_no)

cap.release()
cv2.destroyAllWindows()
print("Data collection complete.")



Prepare for jump clip 1
Starting in 3...
Starting in 2...
Starting in 1...
Recording...
Saved jump clip 1

Prepare for jump clip 2
Starting in 3...
Starting in 2...
Starting in 1...
Recording...
Saved jump clip 2

Prepare for jump clip 3
Starting in 3...
Starting in 2...
Starting in 1...
Recording...
Saved jump clip 3

Prepare for crouch clip 1
Starting in 3...
Starting in 2...
Starting in 1...
Recording...
Saved crouch clip 1

Prepare for crouch clip 2
Starting in 3...
Starting in 2...
Starting in 1...
Recording...
Saved crouch clip 2

Prepare for crouch clip 3
Starting in 3...
Starting in 2...
Starting in 1...
Recording...
Saved crouch clip 3

Prepare for move_left clip 1
Starting in 3...
Starting in 2...
Starting in 1...
Recording...
Saved move_left clip 1

Prepare for move_left clip 2
Starting in 3...
Starting in 2...
Starting in 1...
Recording...
Saved move_left clip 2

Prepare for move_left clip 3
Starting in 3...
Starting in 2...
Starting in 1...
Recording...
Saved move_left cl

### 1. Prepare the dataset for training

In [5]:
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

DATA_DIR = "karate_data_trial"
GESTURES = [
    "jump", "crouch", "move_left", "move_right",
    "high_punch", "low_punch", "strong_kick", "high_kick",
    "hit_combo", "neutral"
]

FRAMES_PER_CLIP = 60
LANDMARKS = 33
DIM = 3

X = []
y = []

for g in GESTURES:
    folder = os.path.join(DATA_DIR, g)
    for file in os.listdir(folder):
        arr = np.load(os.path.join(folder, file))  # (60, 33, 3)

        # Safety check
        if arr.shape != (FRAMES_PER_CLIP, LANDMARKS, DIM):
            print(f"Skipping {file}, shape = {arr.shape}")
            continue

        X.append(arr.reshape(-1))  # 60*33*3 = 5940 features
        y.append(g)

X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)  # (num_samples, 5940)

# Encode labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42
)


X shape: (30, 5940)


### 2. Train a Random Forest classifier

In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import joblib

# 1. Create model
clf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

# 2. FIT THE MODEL (THIS WAS MISSING)
clf.fit(X_train, y_train)

# 3. Predict
y_pred = clf.predict(X_test)

# 4. Evaluation
labels = list(range(len(le.classes_)))

print("Accuracy:", accuracy_score(y_test, y_pred))
print(
    classification_report(
        y_test,
        y_pred,
        labels=labels,
        target_names=le.classes_,
        zero_division=0
    )
)

# 5. Save model
joblib.dump(clf, "karate_rf_model.pkl")
joblib.dump(le, "label_encoder.pkl")


Accuracy: 0.8333333333333334
              precision    recall  f1-score   support

      crouch       0.00      0.00      0.00         0
   high_kick       1.00      1.00      1.00         1
  high_punch       0.00      0.00      0.00         0
   hit_combo       0.00      0.00      0.00         0
        jump       0.00      0.00      0.00         0
   low_punch       1.00      0.50      0.67         2
   move_left       1.00      1.00      1.00         1
  move_right       1.00      1.00      1.00         1
     neutral       0.50      1.00      0.67         1
 strong_kick       0.00      0.00      0.00         0

    accuracy                           0.83         6
   macro avg       0.45      0.45      0.43         6
weighted avg       0.92      0.83      0.83         6



['label_encoder.pkl']

In [7]:
import numpy as np

unique, counts = np.unique(y_train, return_counts=True)
for u, c in zip(unique, counts):
    print(le.inverse_transform([u])[0], c)


crouch 3
high_kick 2
high_punch 3
hit_combo 3
jump 3
low_punch 1
move_left 2
move_right 2
neutral 2
strong_kick 3


### 3. Live gesture detection

In [9]:
import cv2
import mediapipe as mp
import numpy as np
from collections import deque
import joblib

clf = joblib.load("karate_rf_model.pkl")
le = joblib.load("label_encoder.pkl")

mp_pose = mp.solutions.pose
pose = mp_pose.Pose()
cap = cv2.VideoCapture(0)

SEQUENCE_LENGTH = 60
buffer = deque(maxlen=SEQUENCE_LENGTH)
pred_buffer = deque(maxlen=5)

print("Press ESC or Q to quit...")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    res = pose.process(rgb)

    # Always append something
    if res.pose_landmarks:
        lm_list = [[lm.x, lm.y, lm.z] for lm in res.pose_landmarks.landmark]
        mp.solutions.drawing_utils.draw_landmarks(
            frame, res.pose_landmarks, mp_pose.POSE_CONNECTIONS
        )
    else:
        lm_list = np.zeros((33, 3)).tolist()

    buffer.append(lm_list)

    gesture = "..."

    if len(buffer) == SEQUENCE_LENGTH:
        clip_arr = np.array(buffer).reshape(1, -1)

        if clip_arr.shape[1] == clf.n_features_in_:
            pred = clf.predict(clip_arr)[0]
            pred_buffer.append(pred)

            # Majority vote
            gesture = le.inverse_transform(
                [max(set(pred_buffer), key=pred_buffer.count)]
            )[0]

    cv2.putText(frame, gesture, (30, 60),
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 3)

    cv2.imshow("Karate Fighter Gesture Detection", frame)

    key = cv2.waitKey(1)
    if key == 27 or key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


Press ESC or Q to quit...
